In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import os
import pathlib

BASE_DIR = pathlib.Path(os.getcwd())
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

DB_PATH = BASE_DIR / 'data' / 'banco.db'
engine  = create_engine(f'sqlite:///{DB_PATH}', echo=False)

# Carrega tudo do SQLite
df_c   = pd.read_sql('SELECT * FROM tb_clientes', engine)
df_c   = df_c.drop(columns=['cluster_id','cluster_nome'], errors='ignore')
df_cl  = pd.read_sql('SELECT id_cliente, cluster_id, cluster_nome FROM tb_clusters', engine)
df_rfm = pd.read_sql('SELECT * FROM tb_rfm', engine)
df_pro = pd.read_sql('SELECT * FROM tb_propensao', engine)

df = df_c.merge(df_cl,  on='id_cliente', how='left')
df = df.merge(df_rfm,   on='id_cliente', how='left')
df = df.merge(df_pro,   on='id_cliente', how='left')

NOMES_CLUSTERS = {
    0: 'Primeiros Passos',
    1: 'Trajetória Crescente',
    2: 'Potencial Oculto',
    3: 'Self Made',
    4: 'Old Money',
}

DESCRICOES_CLUSTERS = {
    0: 'Jovem iniciando carreira, renda baixa, poucos produtos. Foco em inclusão financeira.',
    1: 'No mercado há anos, relacionamento longo, produtos clássicos. Potencial de cross-sell.',
    2: 'Ganha bem mas pouco explorado em produtos bancários. Alta receptividade a investimentos.',
    3: 'Alta renda, tem produtos mas precisa de mais. Foco em imobiliário e câmbio.',
    4: 'Carteira consolidada. Foco exclusivo em upgrades e produtos premium.',
}

print(f"Clientes: {len(df):,} | Colunas: {len(df.columns)}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"cluster_nome únicos: {df['cluster_nome'].unique()}")

Clientes: 300,000 | Colunas: 58
Missing values: 0
cluster_nome únicos: <ArrowStringArray>
['Trajetória Crescente',     'Potencial Oculto',     'Primeiros Passos',
            'Self Made',            'Old Money']
Length: 5, dtype: str


In [2]:
RECOMENDACOES_NOVOS = {
    0: [  # Primeiros Passos
        ('tem_cartao_credito',  'Cartão de Crédito',   'Seu primeiro cartão com limite adequado ao perfil'),
        ('tem_investimento',    'CDB Digital',          'Investimento inicial a partir de R$ 50'),
        ('tem_seguro',          'Seguro de Vida',       'Proteção acessível para sua família'),
        ('tem_credito_pessoal', 'Crédito Pessoal',      'Crédito pré-aprovado com taxa especial'),
    ],
    1: [  # Trajetória Crescente
        ('tem_investimento',         'Investimento',         'Diversifique sua renda com rentabilidade acima da poupança'),
        ('tem_previdencia',          'Previdência Privada',  'Complemento ideal para sua aposentadoria'),
        ('tem_credito_imobiliario',  'Crédito Imobiliário',  'Financie seu imóvel com as melhores taxas'),
        ('tem_consorcio',            'Consórcio',            'Planejamento para bens de alto valor'),
    ],
    2: [  # Potencial Oculto
        ('tem_previdencia',          'Previdência Privada',  'PGBL com benefício fiscal para sua faixa de renda'),
        ('tem_credito_imobiliario',  'Crédito Imobiliário',  'Linha especial para profissionais de alta renda'),
        ('tem_cambio',               'Câmbio Internacional', 'Remessas e viagens com câmbio competitivo'),
        ('tem_consorcio',            'Consórcio Imobiliário','Planejamento estratégico de patrimônio'),
    ],
    3: [  # Self Made
        ('tem_credito_imobiliario',  'Crédito Imobiliário Plus','Linha exclusiva para segundo imóvel'),
        ('tem_cambio',               'Conta Global',            'Conta em dólar para operações internacionais'),
        ('tem_previdencia',          'Previdência Premium',     'PGBL com benefício fiscal máximo'),
        ('tem_consorcio',            'Consórcio Imobiliário',   'Planejamento estratégico de patrimônio'),
    ],
    4: [  # Old Money
        ('tem_consorcio',            'Consórcio Imobiliário',   'Diversificação de patrimônio imobiliário'),
        ('tem_cambio',               'Conta Multi-Moeda',       'Opere em dólar, euro e libra sem tarifas'),
        ('tem_credito_imobiliario',  'Crédito Imobiliário VIP', 'Linha exclusiva Private para alto patrimônio'),
        ('tem_previdencia',          'Previdência VGBL Plus',   'Proteção patrimonial e planejamento sucessório'),
    ],
}

print("Regras de novos produtos definidas!")
for cluster_id, recomendacoes in RECOMENDACOES_NOVOS.items():
    print(f"  Cluster {cluster_id} — {NOMES_CLUSTERS[cluster_id]}: {len(recomendacoes)} recomendações")

Regras de novos produtos definidas!
  Cluster 0 — Primeiros Passos: 4 recomendações
  Cluster 1 — Trajetória Crescente: 4 recomendações
  Cluster 2 — Potencial Oculto: 4 recomendações
  Cluster 3 — Self Made: 4 recomendações
  Cluster 4 — Old Money: 4 recomendações


In [3]:
RECOMENDACOES_UPGRADE = {
    0: [  # Primeiros Passos
        ('tem_cartao_credito', 'upgrade_cartao_black',      'Cartão Gold',              'Você já usa bem seu cartão — hora de evoluir'),
        ('tem_investimento',   'upgrade_investimento_plus', 'Investimento Plus',         'Diversifique com fundos de maior rentabilidade'),
    ],
    1: [  # Trajetória Crescente
        ('tem_investimento',    'upgrade_investimento_plus',   'Carteira Diversificada',   'Acesse fundos exclusivos para seu perfil'),
        ('tem_credito_pessoal', 'upgrade_limite_credito',      'Aumento de Limite',        'Seu histórico garante condições melhores'),
        ('tem_financiamento',   'upgrade_credito_imobiliario', 'Portabilidade Imobiliária','Taxa menor que seu financiamento atual'),
        ('tem_cheque_especial', 'upgrade_limite_cheque',       'Limite Estendido',         'Mais fôlego para o dia a dia'),
    ],
    2: [  # Potencial Oculto
        ('tem_investimento',   'upgrade_investimento_plus',   'Carteira Premium',         'Acesse fundos exclusivos para seu perfil'),
        ('tem_credito_pessoal','upgrade_limite_credito',      'Aumento de Limite',        'Taxas reduzidas pelo seu histórico'),
        ('tem_cambio',         'upgrade_conta_global',        'Conta Global Multi-Moeda', 'Opere em múltiplas moedas sem tarifas'),
    ],
    3: [  # Self Made
        ('tem_cartao_credito', 'upgrade_cartao_black',        'Cartão Black',             'Benefícios exclusivos e limite sem restrição'),
        ('tem_investimento',   'upgrade_investimento_plus',   'Gestão de Patrimônio',     'Assessoria dedicada para sua carteira'),
        ('tem_cambio',         'upgrade_conta_global',        'Conta Global Multi-Moeda', 'Opere em dólar, euro e libra sem tarifas'),
        ('tem_financiamento',  'upgrade_credito_imobiliario', 'Portabilidade Imobiliária','Taxa menor que seu financiamento atual'),
    ],
    4: [  # Old Money
        ('tem_cartao_credito', 'upgrade_cartao_black',        'Cartão Infinite',          'O mais exclusivo da nossa linha'),
        ('tem_investimento',   'upgrade_investimento_plus',   'Private Banking',          'Gestão personalizada do seu patrimônio'),
        ('tem_cambio',         'upgrade_conta_global',        'Conta Global Premium',     'Assessoria cambial dedicada 24h'),
    ],
}

print("Regras de upgrade definidas!")

Regras de upgrade definidas!


In [4]:
def recomendar_cliente(cliente_id, top_n=3):
    cliente      = df[df['id_cliente'] == cliente_id].iloc[0]
    cluster_id   = int(cliente['cluster_id'])
    cluster_nome = NOMES_CLUSTERS[cluster_id]

    novos = []
    for produto_col, produto_nome, justificativa in RECOMENDACOES_NOVOS[cluster_id]:
        if cliente[produto_col] == 0:
            taxa     = df[df['cluster_id'] == cluster_id][produto_col].mean()
            col_prop = f"propensao_{produto_col.replace('tem_','')}"
            propensao = float(cliente.get(col_prop, 0))
            novos.append({
                'produto':     produto_nome,
                'justificativa': justificativa,
                'taxa_cluster':  taxa,
                'propensao':   propensao,
                'explicacao':  f"Clientes {cluster_nome} contratam este produto em {taxa:.0%} dos casos",
            })

    upgrades = []
    for produto_col, upgrade_col, upgrade_nome, justificativa in RECOMENDACOES_UPGRADE[cluster_id]:
        if cliente[produto_col] == 1 and cliente[upgrade_col] == 0:
            taxa = df[df['cluster_id'] == cluster_id][upgrade_col].mean()
            upgrades.append({
                'produto':     upgrade_nome,
                'justificativa': justificativa,
                'taxa_cluster':  taxa,
                'explicacao':  f"{taxa:.0%} dos clientes {cluster_nome} já fizeram este upgrade",
            })

    return {
        'id_cliente':      cliente_id,
        'cluster_id':      cluster_id,
        'cluster_nome':    cluster_nome,
        'descricao_perfil':DESCRICOES_CLUSTERS[cluster_id],
        'nome':            cliente.get('nome',''),
        'profissao':       cliente.get('profissao',''),
        'renda_mensal':    float(cliente['renda_mensal']),
        'score_credito':   int(cliente['score_credito']),
        'idade':           int(cliente['idade']),
        'qtd_produtos':    int(cliente['qtd_produtos']),
        'rfm_segmento':    cliente.get('rfm_segmento',''),
        'rfm_score':       round(float(cliente.get('rfm_score',0)),2),
        'prioridade_ataque':cliente.get('prioridade_ataque',''),
        'novos_produtos':  sorted(novos, key=lambda x: x['propensao'], reverse=True)[:top_n],
        'upgrades':        upgrades[:top_n],
    }

print("Função de recomendação definida!")

Função de recomendação definida!


In [5]:
print("=" * 65)
for cluster_id in range(5):
    cliente_teste = df[df['cluster_id'] == cluster_id]['id_cliente'].iloc[0]
    resultado     = recomendar_cliente(cliente_teste)

    print(f"\nCLIENTE: {resultado['id_cliente']} — {resultado['nome']}")
    print(f"Perfil:   {resultado['cluster_nome']}")
    print(f"Profissão:{resultado['profissao']}")
    print(f"Renda:    R$ {resultado['renda_mensal']:,.2f} | Score: {resultado['score_credito']}")
    print(f"RFM:      {resultado['rfm_segmento']} (score {resultado['rfm_score']})")
    print(f"Prioridade: {resultado['prioridade_ataque']}")

    print(f"\n  NOVOS PRODUTOS (ordenados por propensão):")
    for r in resultado['novos_produtos']:
        print(f"  → {r['produto']} ({r['propensao']:.1f}% propensão)")
        print(f"     {r['explicacao']}")

    print(f"\n  UPGRADES:")
    for u in resultado['upgrades']:
        print(f"  ↑ {u['produto']}")
        print(f"     {u['explicacao']}")
    print("=" * 65)


CLIENTE: CLI000003 — Anthony Jesus
Perfil:   Primeiros Passos
Profissão:Vendedor
Renda:    R$ 2,500.00 | Score: 481
RFM:      Perdidos (score 1.3)
Prioridade: Retenção

  NOVOS PRODUTOS (ordenados por propensão):
  → Cartão de Crédito (42.3% propensão)
     Clientes Primeiros Passos contratam este produto em 48% dos casos
  → Seguro de Vida (14.8% propensão)
     Clientes Primeiros Passos contratam este produto em 13% dos casos
  → CDB Digital (6.4% propensão)
     Clientes Primeiros Passos contratam este produto em 13% dos casos

  UPGRADES:



CLIENTE: CLI000001 — Sr. Pedro Miguel Camargo
Perfil:   Trajetória Crescente
Profissão:Médico Especialista
Renda:    R$ 20,000.00 | Score: 518
RFM:      Champions (score 4.7)
Prioridade: Baixa

  NOVOS PRODUTOS (ordenados por propensão):
  → Investimento (72.9% propensão)
     Clientes Trajetória Crescente contratam este produto em 34% dos casos
  → Consórcio (0.0% propensão)
     Clientes Trajetória Crescente contratam este produto em 17% dos casos

  UPGRADES:
  ↑ Aumento de Limite
     0% dos clientes Trajetória Crescente já fizeram este upgrade
  ↑ Portabilidade Imobiliária
     5% dos clientes Trajetória Crescente já fizeram este upgrade

CLIENTE: CLI000002 — Samuel Costa
Perfil:   Potencial Oculto
Profissão:Executivo de Vendas
Renda:    R$ 6,977.01 | Score: 462
RFM:      Em Risco (score 2.05)
Prioridade: Retenção

  NOVOS PRODUTOS (ordenados por propensão):
  → Previdência Privada (36.7% propensão)
     Clientes Potencial Oculto contratam este produto em 30% dos casos
  → Crédit


CLIENTE: CLI000007 — Sofia Barros
Perfil:   Self Made
Profissão:Diretor Financeiro (CFO)
Renda:    R$ 23,690.46 | Score: 562
RFM:      Clientes Leais (score 3.8)
Prioridade: Média

  NOVOS PRODUTOS (ordenados por propensão):
  → Conta Global (63.5% propensão)
     Clientes Self Made contratam este produto em 43% dos casos
  → Consórcio Imobiliário (0.0% propensão)
     Clientes Self Made contratam este produto em 26% dos casos

  UPGRADES:
  ↑ Gestão de Patrimônio
     58% dos clientes Self Made já fizeram este upgrade
  ↑ Portabilidade Imobiliária
     22% dos clientes Self Made já fizeram este upgrade

CLIENTE: CLI000034 — Theodoro Pinto
Perfil:   Old Money
Profissão:Piloto Internacional
Renda:    R$ 56,202.87 | Score: 681
RFM:      Clientes Leais (score 3.8)
Prioridade: Alta

  NOVOS PRODUTOS (ordenados por propensão):

  UPGRADES:


In [6]:
print("Gerando recomendações vetorizadas...")

def get_primeiro_produto(row):
    cluster_id = int(row['cluster_id'])
    for produto_col, produto_nome, _ in RECOMENDACOES_NOVOS[cluster_id]:
        if row[produto_col] == 0:
            return produto_nome
    return 'Nenhum'

def get_qtd_novos(row):
    cluster_id = int(row['cluster_id'])
    return sum(1 for produto_col, _, _ in RECOMENDACOES_NOVOS[cluster_id]
               if row[produto_col] == 0)

def get_primeiro_upgrade(row):
    cluster_id = int(row['cluster_id'])
    for produto_col, upgrade_col, upgrade_nome, _ in RECOMENDACOES_UPGRADE[cluster_id]:
        if row[produto_col] == 1 and row[upgrade_col] == 0:
            return upgrade_nome
    return 'Nenhum'

def get_qtd_upgrades(row):
    cluster_id = int(row['cluster_id'])
    return sum(1 for produto_col, upgrade_col, _, _ in RECOMENDACOES_UPGRADE[cluster_id]
               if row[produto_col] == 1 and row[upgrade_col] == 0)

df['primeiro_produto_rec'] = df.apply(get_primeiro_produto, axis=1)
df['qtd_novos_produtos']   = df.apply(get_qtd_novos, axis=1)
df['primeiro_upgrade_rec'] = df.apply(get_primeiro_upgrade, axis=1)
df['qtd_upgrades_rec']     = df.apply(get_qtd_upgrades, axis=1)

df_recomendacoes = df[[
    'id_cliente', 'cluster_id', 'cluster_nome',
    'qtd_novos_produtos', 'qtd_upgrades_rec',
    'primeiro_produto_rec', 'primeiro_upgrade_rec'
]].copy()

with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS tb_recomendacoes"))
    conn.commit()

df_recomendacoes.to_sql('tb_recomendacoes', engine, if_exists='replace', index=False)

print("Recomendações geradas!")
print(df_recomendacoes.groupby('cluster_nome')[
    ['qtd_novos_produtos','qtd_upgrades_rec']
].mean().round(2))

Gerando recomendações vetorizadas...


Recomendações geradas!
                      qtd_novos_produtos  qtd_upgrades_rec
cluster_nome                                              
Old Money                           1.30              0.80
Potencial Oculto                    3.19              0.88
Primeiros Passos                    3.10              0.61
Self Made                           2.23              1.35
Trajetória Crescente                3.10              1.15


In [7]:
print("=== DIAGNÓSTICO FINAL 07 ===\n")

with engine.connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM tb_recomendacoes")).scalar()
    print(f"tb_recomendacoes: {total:,} registros")

    print("\nMédia de recomendações por cluster:")
    for row in conn.execute(text("""
        SELECT cluster_nome,
               ROUND(AVG(qtd_novos_produtos),2) as media_novos,
               ROUND(AVG(qtd_upgrades_rec),2) as media_upgrades
        FROM tb_recomendacoes
        GROUP BY cluster_nome
        ORDER BY media_novos DESC
    """)):
        print(f"  {row[0]:22} | novos: {row[1]} | upgrades: {row[2]}")

print(f"\nTabelas no banco:")
with engine.connect() as conn:
    for row in conn.execute(text("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")):
        count = conn.execute(text(f"SELECT COUNT(*) FROM {row[0]}")).scalar()
        print(f"  {row[0]:25} | {count:,} registros")

=== DIAGNÓSTICO FINAL 07 ===

tb_recomendacoes: 300,000 registros

Média de recomendações por cluster:
  Potencial Oculto       | novos: 3.19 | upgrades: 0.88
  Trajetória Crescente   | novos: 3.1 | upgrades: 1.15
  Primeiros Passos       | novos: 3.1 | upgrades: 0.61
  Self Made              | novos: 2.23 | upgrades: 1.35
  Old Money              | novos: 1.3 | upgrades: 0.8

Tabelas no banco:


  tb_clientes               | 300,000 registros
  tb_clusters               | 300,000 registros
  tb_depara                 | 5 registros
  tb_depara_clientes        | 300,000 registros
  tb_estados_cidades        | 64 registros
  tb_ibge_salarios          | 69 registros
  tb_perfil_clusters        | 5 registros
  tb_propensao              | 300,000 registros
  tb_recomendacoes          | 300,000 registros
  tb_rfm                    | 300,000 registros


In [8]:
import os
import requests
import pathlib

BASE_DIR = pathlib.Path(os.getcwd())
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

caminho_geo = BASE_DIR / 'data' / 'raw' / 'brasil_estados.geojson'

if not caminho_geo.exists():
    print("Baixando GeoJSON...")
    url = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"
    resp = requests.get(url, timeout=15)
    with open(caminho_geo, 'w', encoding='utf-8') as f:
        f.write(resp.text)
    print(f"Salvo em: {caminho_geo}")
else:
    print(f"GeoJSON já existe: {caminho_geo}")

# Valida
import json
with open(caminho_geo, 'r', encoding='utf-8') as f:
    geo = json.load(f)

print(f"Features: {len(geo['features'])}")
print(f"Propriedades: {geo['features'][0]['properties']}")

GeoJSON já existe: c:\Users\caleb\Documents\SANTANDER-MOTOR-RECOMENDACAO\data\raw\brasil_estados.geojson


Features: 27
Propriedades: {'id': 1, 'name': 'Acre', 'sigla': 'AC', 'regiao_id': '3', 'codigo_ibg': '12', 'cartodb_id': 1, 'created_at': '2015-02-09T16:46:01Z', 'updated_at': '2015-02-09T16:46:01Z'}
